# Committees Admin Notebook

Manage committee nodes: **ATIWorkingGroup**.

ATI Working Groups are specialized teams responsible for overseeing and implementing specific aspects of the Accessible Technology Initiative. The three working groups are: **Web**, **Instructional Materials**, and **Procurement**. Each is responsible for goals and success indicators in their domain.

## Connection Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))

from app.database.graph_schema import set_connection
set_connection()

import pandas as pd
from datetime import date, datetime
pd.set_option('display.max_colwidth', 80)
print('Connection established.')

---
## READ Operations

### List All Working Groups

In [ ]:
from app.database.queries.committees.read import get_all_committees

committees = get_all_committees()
if committees:
    df = pd.DataFrame([{
        'unique_id': c.unique_id,
        'name': c.name,
        'description': c.description
    } for c in committees])
    display(df)
else:
    print('No ATIWorkingGroup nodes found.')

### List Working Group Members

Shows all persons assigned to each working group via the `participates_in` relationship.

In [ ]:
from app.database.graph_schema import ATIWorkingGroup, Person
from neomodel import db

committees = ATIWorkingGroup.nodes.all()
for committee in committees:
    # Find persons who participate in this working group
    query = """
    MATCH (p:Person)-[:participates_in]->(wg:ATIWorkingGroup {name: $name})
    RETURN p
    """
    results, _ = db.cypher_query(query, {'name': committee.name})
    members = [Person.inflate(row[0]) for row in results]
    
    print(f'\n=== {committee.name} ({len(members)} members) ===')
    if members:
        df = pd.DataFrame([{
            'name': m.name,
            'employee_id': m.employee_id,
            'title': m.title,
            'email': m.email,
            'active': m.active,
            'ati_role': m.ati_role
        } for m in members])
        display(df)
    else:
        print('  No members assigned.')

### List Goals for Each Working Group

In [ ]:
from app.database.graph_schema import ATIWorkingGroup

committees = ATIWorkingGroup.nodes.all()
for committee in committees:
    goals = committee.responsible_for.all()
    print(f'\n=== {committee.name} ({len(goals)} goals) ===')
    if goals:
        df = pd.DataFrame([g.serialize() for g in goals])
        display(df)
    else:
        print('  No goals assigned.')

### Working Group Summary

Shows member count, goal count, and success indicator count for each working group.

In [ ]:
from app.database.graph_schema import ATIWorkingGroup, Person
from neomodel import db

committees = ATIWorkingGroup.nodes.all()
summary = []
for committee in committees:
    # Count members
    query = "MATCH (p:Person)-[:participates_in]->(wg:ATIWorkingGroup {name: $name}) RETURN count(p)"
    results, _ = db.cypher_query(query, {'name': committee.name})
    member_count = results[0][0] if results else 0
    
    # Count goals and indicators
    goals = committee.responsible_for.all()
    indicator_count = sum(len(g.supporting_success_indicators.all()) for g in goals)
    
    summary.append({
        'name': committee.name,
        'members': member_count,
        'goals': len(goals),
        'success_indicators': indicator_count
    })

if summary:
    display(pd.DataFrame(summary))
else:
    print('No working groups found.')

---
## CREATE Operations

No dedicated create functions exist for ATIWorkingGroup. The three working groups (Web, Instructional Materials, Procurement) are typically pre-seeded. To create a new one manually:

In [ ]:
from app.database.graph_schema import ATIWorkingGroup

# new_wg = ATIWorkingGroup(
#     name='New Working Group Name',
#     description='Description of the working group'
# )
# new_wg.save()
# print(f'Created: {new_wg.name}')

---
## UPDATE Operations

### Add a Person to a Working Group

Parameters:
- `employee_id` (str) - The employee ID of the person
- `committee_name` (str) - The name of the working group (e.g., `"Web"`, `"Instructional Materials"`, `"Procurement"`)

In [ ]:
from app.database.queries.committees.update import add_person_to_committee

# result = add_person_to_committee(
#     employee_id='123456789',
#     committee_name='Web'
# )
# print(result)

### Update a Working Group's Description

In [ ]:
from app.database.graph_schema import ATIWorkingGroup

wg_name = 'Web'  # <-- change this

# wg = ATIWorkingGroup.nodes.get(name=wg_name)
# wg.description = 'Updated description for the working group'
# wg.save()
# print(f'Updated: {wg.name}')

---
## DELETE Operations

### Remove a Person from a Working Group

In [ ]:
from app.database.graph_schema import Person, ATIWorkingGroup

# person = Person.nodes.get(employee_id='123456789')  # <-- change
# wg = ATIWorkingGroup.nodes.get(name='Web')  # <-- change

# if person.in_ati_working_group.is_connected(wg):
#     person.in_ati_working_group.disconnect(wg)
#     print(f'Removed {person.name} from {wg.name}')
# else:
#     print('Person is not in this working group.')

### Delete a Working Group

Use with extreme caution. This will remove the working group and all its relationships.

In [ ]:
from app.database.graph_schema import ATIWorkingGroup
from neomodel import db

# wg_name = 'Working Group Name'  # <-- change
# wg = ATIWorkingGroup.nodes.get(name=wg_name)
# db.cypher_query('MATCH (n) WHERE elementId(n) = $eid DETACH DELETE n', {'eid': wg.element_id})
# print(f'Deleted working group: {wg_name}')

print('Uncomment the delete operation you need and run again.')

---
## RELATIONSHIP Management

### View All Relationships for a Working Group

In [ ]:
from app.database.graph_schema import ATIWorkingGroup, Person
from neomodel import db

wg_name = 'Web'  # <-- change this: Web, Instructional Materials, Procurement

try:
    wg = ATIWorkingGroup.nodes.get(name=wg_name)
    print(f'=== {wg.name} ===')
    print(f'  Description: {wg.description}')
    print(f'  unique_id: {wg.unique_id}')
    
    # Members (reverse of participates_in)
    query = "MATCH (p:Person)-[:participates_in]->(wg:ATIWorkingGroup {name: $name}) RETURN p"
    results, _ = db.cypher_query(query, {'name': wg_name})
    members = [Person.inflate(row[0]) for row in results]
    print(f'\nMembers ({len(members)}):')
    for m in members:
        print(f'  - {m.name} ({m.employee_id}) - {m.title}')
    
    # Goals
    goals = wg.responsible_for.all()
    print(f'\nGoals ({len(goals)}):')
    for g in goals:
        indicators = g.supporting_success_indicators.all()
        print(f'  - Goal #{g.goal_number}: {g.name} ({len(indicators)} indicators)')
        for si in indicators:
            print(f'      {si.composite_key}: {si.success_indicator}')
except Exception as e:
    print(f'Error: {e}')

### Connect a Goal to a Working Group

In [ ]:
from app.database.graph_schema import ATIWorkingGroup, Goal

# wg = ATIWorkingGroup.nodes.get(name='Web')  # <-- change
# goal = Goal.nodes.get(goal_number=1)  # <-- change

# if not wg.responsible_for.is_connected(goal):
#     wg.responsible_for.connect(goal)
#     print(f'Connected {wg.name} -> Goal #{goal.goal_number}')
# else:
#     print('Already connected.')

### Disconnect a Goal from a Working Group

In [ ]:
from app.database.graph_schema import ATIWorkingGroup, Goal

# wg = ATIWorkingGroup.nodes.get(name='Web')  # <-- change
# goal = Goal.nodes.get(goal_number=1)  # <-- change

# if wg.responsible_for.is_connected(goal):
#     wg.responsible_for.disconnect(goal)
#     print(f'Disconnected {wg.name} from Goal #{goal.goal_number}')
# else:
#     print('Not connected.')